In [35]:
!pip install langchain-groq
!pip install python-dotenv


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [37]:
load_dotenv()

True

In [49]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq

model = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)


In [50]:
#create a state
from typing import TypedDict
class LLMState(TypedDict):
    question: str
    answer: str

In [51]:
def llm_qa(state: LLMState) -> LLMState:
    # extract the question from state
    question = state["question"]

    # form a proper prompt
    prompt = f"Answer the following question: {question}"

    # ask question to the LLM
    response = model.invoke(prompt)
    answer = response.content     # correct for ChatGroq

    # update the state
    state["answer"] = answer

    return state


In [52]:
#create our graph
graph= StateGraph(LLMState)
#add nodes
graph.add_node('llm_qa', llm_qa)

## add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

#compile
workflow =graph.compile()

In [53]:
#execute
intial_state= {'question': 'How far is moon from the earth'}
final_state = workflow.invoke(intial_state)
print(final_state)

{'question': 'How far is moon from the earth', 'answer': "The average distance from the Earth to the Moon is approximately 384,400 kilometers (238,900 miles). This distance is constantly changing due to the elliptical shape of the Moon's orbit around the Earth. At its closest point (called perigee), the Moon is about 363,300 kilometers (225,300 miles) away, and at its farthest point (apogee), it is about 405,500 kilometers (252,000 miles) away."}


In [54]:
model.invoke('How far is moon from the earth?')

AIMessage(content="The average distance from the Earth to the Moon is approximately 384,400 kilometers (238,900 miles). This distance is constantly changing due to the elliptical shape of the Moon's orbit around the Earth. At its closest point (called perigee), the Moon is about 363,300 kilometers (225,300 miles) away, and at its farthest point (apogee), it is about 405,500 kilometers (252,000 miles) away.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 43, 'total_tokens': 140, 'completion_time': 0.10117069, 'completion_tokens_details': None, 'prompt_time': 0.003096748, 'prompt_tokens_details': None, 'queue_time': 0.051149432, 'total_time': 0.104267438}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e750f72ec9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--9ea9db0c-807a-4c6f-bb87-9cecb47b8014-0', usage_metadata={'input_tokens': 43, 'output_tokens': 97